In [ ]:
import math
from typing import Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import pearsonr
from torch.utils.data import DataLoader, TensorDataset
import copy
from typing import Any
import matplotlib.pyplot as plt


class AttentionHead(nn.Module):
    def __init__(
        self,
        d_model: int,
        head_dim: int,
        dropout: float = 0.0,
        attention_type: str = "full",
        attn_window: Optional[int] = None,
    ):
        super().__init__()
        self.q_proj = nn.Linear(d_model, head_dim)
        self.k_proj = nn.Linear(d_model, head_dim)
        self.v_proj = nn.Linear(d_model, head_dim)
        self.dropout = nn.Dropout(dropout)
        self.attention_type = attention_type
        self.attn_window = attn_window

    def _build_mask(self, t: int, device: torch.device) -> Optional[torch.Tensor]:
        if self.attention_type == "full":
            return None

        idx = torch.arange(t, device=device)
        col = idx.view(1, -1)
        row = idx.view(-1, 1)

        if self.attention_type == "causal":
            # Allow attention to current and previous time bins only.
            invalid = col > row
        elif self.attention_type == "local":
            if self.attn_window is None:
                raise ValueError("attention_type='local' requires attn_window")
            invalid = (col > row) | ((row - col) >= int(self.attn_window))
        else:
            raise ValueError(f"Unknown attention_type: {self.attention_type}")

        return invalid

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, time, d_model)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        scale = math.sqrt(q.size(-1))
        scores = torch.bmm(q, k.transpose(1, 2)) / scale
        mask = self._build_mask(scores.size(-1), scores.device)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(0), float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.bmm(attn, v)
        return out


class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int = 4,
        dropout: float = 0.0,
        attention_type: str = "full",
        attn_window: Optional[int] = None,
    ):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError(
                f"d_model ({d_model}) must be divisible by n_heads ({n_heads})."
            )

        head_dim = d_model // n_heads
        self.heads = nn.ModuleList(
            [
                AttentionHead(
                    d_model=d_model,
                    head_dim=head_dim,
                    dropout=dropout,
                    attention_type=attention_type,
                    attn_window=attn_window,
                )
                for _ in range(n_heads)
            ]
        )
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        head_outs = [h(x) for h in self.heads]
        concat = torch.cat(head_outs, dim=-1)
        out = self.out_proj(concat)
        return self.dropout(out)


class AttentionBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int = 4,
        ff_mult: int = 4,
        dropout: float = 0.1,
        attention_type: str = "full",
        attn_window: Optional[int] = None,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
            attention_type=attention_type,
            attn_window=attn_window,
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class NeuralAttentionRegressor(nn.Module):
    """
    Maps behavioral regressors to neural targets.

    Expected shape:
      x: (batch, time, input_dim)
      y: (batch, time, output_dim)
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        d_model: int = 256,
        n_heads: int = 8,
        n_layers: int = 3,
        ff_mult: int = 4,
        dropout: float = 0.1,
        use_positional_encoding: bool = False,
        attention_type: str = "full",
        attn_window: Optional[int] = None,
    ):
        super().__init__()
        self.use_positional_encoding = use_positional_encoding
        self.in_proj = nn.Linear(input_dim, d_model)
        self.blocks = nn.ModuleList(
            [
                AttentionBlock(
                    d_model=d_model,
                    n_heads=n_heads,
                    ff_mult=ff_mult,
                    dropout=dropout,
                    attention_type=attention_type,
                    attn_window=attn_window,
                )
                for _ in range(n_layers)
            ]
        )
        self.norm = nn.LayerNorm(d_model)
        self.out_proj = nn.Linear(d_model, output_dim)

    def _sinusoidal_positional_encoding(
        self, t: int, d: int, device: torch.device
    ) -> torch.Tensor:
        pos = torch.arange(t, device=device, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d, 2, device=device, dtype=torch.float32)
            * (-math.log(10000.0) / d)
        )
        pe = torch.zeros(t, d, device=device)
        pe[:, 0::2] = torch.sin(pos * div)
        if d > 1:
            pe[:, 1::2] = torch.cos(pos * div[: (d // 2)])
        return pe

    def forward(self, x: torch.Tensor):
        h = self.in_proj(x)
        if self.use_positional_encoding:
            pe = self._sinusoidal_positional_encoding(h.size(1), h.size(2), h.device)
            h = h + pe.unsqueeze(0)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        y_hat = self.out_proj(h)
        return y_hat, h


def make_dataloaders(
    x: torch.Tensor,
    y: torch.Tensor,
    val_fraction: float = 0.2,
    batch_size: int = 32,
    shuffle_train: bool = True,
) -> Dict[str, DataLoader]:
    if x.ndim != 3 or y.ndim != 3:
        raise ValueError(
            f"Expected 3D tensors (batch,time,feat), got X={tuple(x.shape)} Y={tuple(y.shape)}"
        )
    if x.shape[:2] != y.shape[:2]:
        raise ValueError(
            f"Batch/time dims must match, got X={tuple(x.shape)} Y={tuple(y.shape)}"
        )
    if not (0.0 <= val_fraction < 1.0):
        raise ValueError("val_fraction must be in [0,1)")

    n = x.size(0)
    n_val = int(round(n * val_fraction))
    n_val = min(max(1, n_val), n - 1) if n > 1 and val_fraction > 0 else 0

    if n_val > 0:
        x_train, y_train = x[:-n_val], y[:-n_val]
        x_val, y_val = x[-n_val:], y[-n_val:]
    else:
        x_train, y_train = x, y
        x_val, y_val = None, None

    train_loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=min(batch_size, len(x_train)),
        shuffle=shuffle_train,
    )

    val_loader = None
    if x_val is not None:
        val_loader = DataLoader(
            TensorDataset(x_val, y_val),
            batch_size=min(batch_size, len(x_val)),
            shuffle=False,
        )

    return {"train": train_loader, "val": val_loader}


def pearson_r_flat(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Match sweep metric: flattened Pearson r with constant-signal guards."""
    y_true_np = y_true.detach().reshape(-1).float().cpu().numpy()
    y_pred_np = y_pred.detach().reshape(-1).float().cpu().numpy()
    if y_true_np.size < 2 or y_pred_np.size < 2:
        return float("nan")
    if np.std(y_true_np) <= 1e-12 or np.std(y_pred_np) <= 1e-12:
        return float("nan")
    return float(pearsonr(y_true_np, y_pred_np).statistic)


def evaluate_model(
    model: nn.Module, data_loader: DataLoader, device: torch.device
) -> Dict[str, float]:
    model.eval()
    losses = []
    r_vals = []

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            out, _ = model(x_batch)
            loss = F.mse_loss(out, y_batch)
            losses.append(loss.item())
            r_vals.append(pearson_r_flat(y_batch, out))

    return {
        "loss": float(sum(losses) / max(1, len(losses))),
        "pearson_r": float(sum(r_vals) / max(1, len(r_vals))),
    }


def train_attention_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: Optional[DataLoader] = None,
    epochs: int = 250,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    grad_clip: float = 1.0,
    print_every: int = 25,
    device: str = "cuda",
) -> Dict[str, list]:
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    hist = {
        "train_loss": [],
        "train_pearson_r": [],
        "val_loss": [],
        "val_pearson_r": [],
    }

    best_val_r = -float("inf")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_r = 0.0
        n_batches = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            out, _ = model(x_batch)
            loss = F.mse_loss(out, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            running_loss += loss.item()
            running_r += pearson_r_flat(y_batch, out)
            n_batches += 1

        train_loss = running_loss / max(1, n_batches)
        train_r = running_r / max(1, n_batches)
        hist["train_loss"].append(train_loss)
        hist["train_pearson_r"].append(train_r)

        if val_loader is not None:
            val_metrics = evaluate_model(model, val_loader, device)
            hist["val_loss"].append(val_metrics["loss"])
            hist["val_pearson_r"].append(val_metrics["pearson_r"])
            if val_metrics["pearson_r"] > best_val_r:
                best_val_r = val_metrics["pearson_r"]
                best_state = {
                    k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                }

        if ep % print_every == 0 or ep == 1 or ep == epochs:
            if val_loader is not None:
                print(
                    f"epoch {ep:4d} | train_loss {train_loss:.6f} | train_r {train_r:.4f} "
                    f"| val_loss {hist['val_loss'][-1]:.6f} | val_r {hist['val_pearson_r'][-1]:.4f}"
                )
            else:
                print(
                    f"epoch {ep:4d} | train_loss {train_loss:.6f} | train_r {train_r:.4f}"
                )

    if best_state is not None:
        model.load_state_dict(best_state)

    return hist


def predict_attention_model(
    model: nn.Module, x: torch.Tensor, device: str = "cuda"
) -> torch.Tensor:
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)
    with torch.no_grad():
        y_hat, _ = model(x.to(device))
    return y_hat.cpu()


# Training/evaluation utilities (aligned with sweep metric: Pearson r)


def evaluate_attention_model(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> Dict[str, float]:
    model.eval()
    losses = []
    pearsons = []

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            out, _ = model(x_batch)
            loss = F.mse_loss(out, y_batch)

            losses.append(loss.item())
            pearsons.append(pearson_r_flat(y_batch, out))

    return {
        "loss": float(sum(losses) / max(1, len(losses))),
        "pearson_r": float(sum(pearsons) / max(1, len(pearsons))),
    }


def train_attention_regressor(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: Optional[DataLoader] = None,
    epochs: int = 250,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    grad_clip: float = 1.0,
    patience: Optional[int] = 50,
    print_every: int = 25,
    device: str = "cuda",
) -> Dict[str, Any]:
    """Dedicated training loop for sequence-to-sequence neural regression."""
    dev = torch.device(device if torch.cuda.is_available() else "cpu")
    model.to(dev)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    hist = {
        "train_loss": [],
        "train_pearson_r": [],
        "val_loss": [],
        "val_pearson_r": [],
    }

    best_val_pearson = -float("inf")
    best_state = None
    best_epoch = 0
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_pearson = 0.0
        n_batches = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(dev)
            y_batch = y_batch.to(dev)

            optimizer.zero_grad()
            out, _ = model(x_batch)
            loss = F.mse_loss(out, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            running_loss += loss.item()
            running_pearson += pearson_r_flat(y_batch, out)
            n_batches += 1

        train_loss = running_loss / max(1, n_batches)
        train_pearson = running_pearson / max(1, n_batches)

        hist["train_loss"].append(train_loss)
        hist["train_pearson_r"].append(train_pearson)

        if val_loader is not None:
            val_metrics = evaluate_attention_model(model, val_loader, dev)
            hist["val_loss"].append(val_metrics["loss"])
            hist["val_pearson_r"].append(val_metrics["pearson_r"])

            if (
                not np.isnan(val_metrics["pearson_r"])
                and val_metrics["pearson_r"] > best_val_pearson
            ):
                best_val_pearson = val_metrics["pearson_r"]
                best_epoch = ep
                bad_epochs = 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_epochs += 1

        if ep % print_every == 0 or ep == 1 or ep == epochs:
            if val_loader is not None:
                print(
                    f"epoch {ep:4d} | "
                    f"train_loss {train_loss:.6f} train_pearson_r {train_pearson:.4f} | "
                    f"val_loss {hist['val_loss'][-1]:.6f} val_pearson_r {hist['val_pearson_r'][-1]:.4f}"
                )
            else:
                print(
                    f"epoch {ep:4d} | train_loss {train_loss:.6f} train_pearson_r {train_pearson:.4f}"
                )

        if val_loader is not None and patience is not None and bad_epochs >= patience:
            print(
                f"Early stopping at epoch {ep} (no val Pearson-r improvement for {patience} epochs)"
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "history": hist,
        "best_val_pearson_r": best_val_pearson,
        "best_epoch": best_epoch,
    }


# Plotting utilities (loss + Pearson r)


def plot_training_curves(
    history: Dict[str, list], title_prefix: str = "Attention Regressor"
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(history.get("train_loss", []), label="train")
    if history.get("val_loss"):
        axes[0].plot(history.get("val_loss", []), label="val")
    axes[0].set_title(f"{title_prefix} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE")
    axes[0].legend()

    # Pearson r
    axes[1].plot(history.get("train_pearson_r", []), label="train")
    if history.get("val_pearson_r"):
        axes[1].plot(history.get("val_pearson_r", []), label="val")
    axes[1].set_title(f"{title_prefix} - Pearson r")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("r")
    axes[1].set_ylim(-1.0, 1.0)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


# Data loading is centralized in src/sweep.py (load_dmat_timecourse)
# to keep dmat['X']/dmat['Y'] handling consistent across notebooks.

# Train/val split helper aligned with sweep schema


def make_trialwise_dataloaders(
    x_trials_np: np.ndarray,
    y_trials_np: np.ndarray,
    val_fraction: float = 0.2,
    min_val_trials: int = 50,
    batch_size: int = 32,
    device: str = "cuda",
):
    if x_trials_np.ndim != 3 or y_trials_np.ndim != 3:
        raise ValueError(
            f"Expected (n_trials,n_bins,n_feat) and (n_trials,n_bins,n_targets), got {x_trials_np.shape} {y_trials_np.shape}"
        )
    if x_trials_np.shape[:2] != y_trials_np.shape[:2]:
        raise ValueError(
            f"X/Y trial-bin mismatch: {x_trials_np.shape} vs {y_trials_np.shape}"
        )

    n_trials = x_trials_np.shape[0]
    if n_trials < 2:
        raise ValueError(f"Need >=2 trials, got {n_trials}")
    if not (0.0 <= val_fraction < 1.0):
        raise ValueError("val_fraction must be in [0,1)")

    if val_fraction > 0:
        n_val = max(min_val_trials, int(round(n_trials * val_fraction)))
        n_val = min(max(1, n_val), n_trials - 1)
        train_end = n_trials - n_val
        x_train_np, y_train_np = x_trials_np[:train_end], y_trials_np[:train_end]
        x_val_np, y_val_np = x_trials_np[train_end:], y_trials_np[train_end:]
    else:
        x_train_np, y_train_np = x_trials_np, y_trials_np
        x_val_np, y_val_np = None, None

    x_train = torch.tensor(x_train_np, dtype=torch.float32)
    y_train = torch.tensor(y_train_np, dtype=torch.float32)
    x_val = (
        torch.tensor(x_val_np, dtype=torch.float32) if x_val_np is not None else None
    )
    y_val = (
        torch.tensor(y_val_np, dtype=torch.float32) if y_val_np is not None else None
    )

    train_loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=min(batch_size, len(x_train)),
        shuffle=True,
    )

    val_loader = None
    if x_val is not None:
        val_loader = DataLoader(
            TensorDataset(x_val, y_val),
            batch_size=min(batch_size, len(x_val)),
            shuffle=False,
        )

    print(
        f"train trials={len(x_train)} | val trials={0 if x_val is None else len(x_val)} | batch_size={batch_size}"
    )
    return train_loader, val_loader

In [ ]:
# Attention-head sweep with CSV logging (incremental writes per run)
import time
import sys
import random
from pathlib import Path

# Make sure src/ is importable when notebook runs from notebooks/ or repo root
cwd = Path.cwd().resolve()
repo = cwd if (cwd / "src").exists() else cwd.parent
src_path = repo / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from sweep import (
    generate_attention_search_configs,
    run_custom_sweep,
    load_dmat_timecourse,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# ---- Core run settings ----
SESSION = "20231211_172819"  # or "20231225_123125"
VAL_FRACTION = 0.2
MIN_VAL_TRIALS = 50
PRINT_EVERY = 100
MAX_CONFIGS = 15  # lighter local run; set None for full grid
RESULTS_CSV = str((repo / "results" / "mha_attention_sweep.csv").resolve())

# ---- Requested constraints + conservative defaults ----
ATTN_ARCH_GRID = {
    "d_model": [128],  # lighter local search
    "n_heads": [1, 2],  # lighter local search
    "n_layers": [1],  # lighter local search
    "ff_mult": [2, 4],
    "dropout": [0.0, 0.1, 0.2],
    "use_positional_encoding": [False, True],
    "attention_type": ["full", "causal"],
    "attn_window": [None],  # local attention deferred to VM heavy sweep
}

ATTN_TRAIN_GRID = {
    "epochs": [250],  # requested: fixed 250
    "batch_size": [16, 32, 64, 128, 256],  # requested range
    "lr": [5e-4, 1e-3],
    "weight_decay": [0.0, 1e-4],
    "grad_clip": [1.0, 2.0],
    "patience": [20],
}

x_trials, y_trials, used_dmat_path = load_dmat_timecourse(SESSION, repo_root=repo)
print(f"Loaded {used_dmat_path.name}: X={x_trials.shape}, Y={y_trials.shape}")
print(f"Session={SESSION} | DMAT={used_dmat_path.name} | device={DEVICE}")

configs = generate_attention_search_configs(
    max_configs=MAX_CONFIGS,
    seed=SEED,
    arch_grid=ATTN_ARCH_GRID,
    training_grid=ATTN_TRAIN_GRID,
)
print(f"Attention sweep configs: {len(configs)}")


def run_mha_config(cfg):
    """Build dataloaders/model, train one config, and return sweep-compatible metrics."""
    train_loader, val_loader = make_trialwise_dataloaders(
        x_trials,
        y_trials,
        val_fraction=VAL_FRACTION,
        min_val_trials=MIN_VAL_TRIALS,
        batch_size=int(cfg["batch_size"]),
        device=DEVICE,
    )

    model = NeuralAttentionRegressor(
        input_dim=x_trials.shape[-1],
        output_dim=y_trials.shape[-1],
        d_model=int(cfg["d_model"]),
        n_heads=int(cfg["n_heads"]),
        n_layers=int(cfg["n_layers"]),
        ff_mult=int(cfg["ff_mult"]),
        dropout=float(cfg["dropout"]),
        use_positional_encoding=bool(cfg["use_positional_encoding"]),
        attention_type=str(cfg["attention_type"]),
        attn_window=None if cfg.get("attn_window") is None else int(cfg["attn_window"]),
    )

    t0 = time.perf_counter()
    out = train_attention_regressor(
        model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=int(cfg["epochs"]),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
        grad_clip=float(cfg["grad_clip"]),
        patience=int(cfg["patience"]),
        print_every=PRINT_EVERY,
        device=DEVICE,
    )
    elapsed = time.perf_counter() - t0

    hist = out["history"]
    metric_hist = (
        hist["val_pearson_r"]
        if len(hist["val_pearson_r"]) > 0
        else hist["train_pearson_r"]
    )
    loss_hist = hist["val_loss"] if len(hist["val_loss"]) > 0 else hist["train_loss"]

    final_metric = float(metric_hist[-1]) if len(metric_hist) > 0 else float("nan")
    final_loss = float(loss_hist[-1]) if len(loss_hist) > 0 else float("nan")

    return {
        "loss_hist": [float(x) for x in loss_hist],
        "metric_hist": [float(x) for x in metric_hist],
        "best_metric": float(out["best_val_pearson_r"]),
        "best_epoch": int(out["best_epoch"]),
        "final_metric": final_metric,
        "final_loss": final_loss,
        "elapsed_sec": round(elapsed, 2),
        "model_params": int(sum(p.numel() for p in model.parameters())),
    }


df_new = run_custom_sweep(
    configs=configs,
    run_config_fn=run_mha_config,
    results_csv=RESULTS_CSV,
    static_fields={
        "session": SESSION,
        "dmat_path": str(used_dmat_path),
        "val_fraction": VAL_FRACTION,
        "min_val_trials": MIN_VAL_TRIALS,
        "device": DEVICE,
    },
)

print(f"\nSaved rows this run: {len(df_new)}")
print(f"Results CSV: {RESULTS_CSV}")

# quick top-k summary
display_cols = [
    "run_id",
    "session",
    "best_metric",
    "best_epoch",
    "final_metric",
    "final_loss",
    "elapsed_sec",
    "batch_size",
    "d_model",
    "n_heads",
    "n_layers",
    "ff_mult",
    "dropout",
    "use_positional_encoding",
    "attention_type",
    "attn_window",
    "lr",
    "weight_decay",
    "grad_clip",
    "patience",
    "model_params",
]
print("\nTop 5 by best validation Pearson r")
display(df_new.sort_values("best_metric", ascending=False)[display_cols].head(5))